
# Minimal RAG PoC — Clean Version (Local Embeddings + FAISS, Ollama Generation)

- **Embeddings**: local (`sentence-transformers/all-MiniLM-L6-v2`) for both documents & queries
- **Index**: FAISS (inner product on L2-normalized vectors)
- **Retrieval**: top-k chunk search
- **Eval**: expected-substring check + citation sanity (citations must reference retrieved chunks)


## 1) Setup

In [2]:

# If needed, install once:
# %pip install --upgrade sentence-transformers faiss-cpu numpy pandas tqdm openai
# For local LLM via Ollama:
# %pip install --upgrade requests
# Vector store
# %pip install --upgrade chromadb


## 2) Imports & configuration

In [3]:
#from dotenv import load_dotenv
#load_dotenv()

import os
import re
from typing import List, Tuple

import numpy as np
import pandas as pd
from tqdm import tqdm
# Local embeddings
from sentence_transformers import SentenceTransformer

# Local LLM via Ollama (optional)
import requests

# ----- Config ----- NOT USED with Ollama
EMBED_MODEL_LOCAL = "sentence-transformers/all-MiniLM-L6-v2"
#GEN_MODEL = "gpt-4o-mini"

CHUNK_SIZE = 400
CHUNK_OVERLAP = 60
TOP_K = 4
#TEMPERATURE = 0.2

# Init embedding model (docs + queries)

embedder = SentenceTransformer(EMBED_MODEL_LOCAL)  # used for docs + queries

# ----- Local LLM (Ollama) config -----
# Prereqs:
# 1) Install Ollama: https://ollama.com
# 2) Pull a model, e.g.:  ollama pull llama3.1:8b
# 3) Ensure Ollama is running (default: http://localhost:11434)
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.1:8b")


# ChromaDB
import chromadb

# ----- ChromaDB config -----
# Use persistent storage so the index survives kernel restarts.
CHROMA_PERSIST_DIR = os.getenv("CHROMA_PERSIST_DIR", "./chroma_db")
CHROMA_COLLECTION = os.getenv("CHROMA_COLLECTION", "amber_messages")


ModuleNotFoundError: No module named 'numpy'

In [ ]:
#import os
#print("OPENAI_API_KEY =", os.getenv("OPENAI_API_KEY"))

## 3) Sample documents (~10)

In [ ]:

docs = [
    ("Astronomy Notes", 
     "Stars form in molecular clouds. The lifecycle of a star depends on its mass. "
     "Massive stars end as supernovae, while Sun-like stars become white dwarfs."),

    ("Home Coffee Guide", 
     "Use freshly roasted beans. Grind size affects extraction: finer for espresso, coarser for French press. "
     "Water temperature around 92-96°C usually works well."),

    ("Indoor Plants 101", 
     "Snake plants tolerate low light and infrequent watering. Peace lilies like indirect light and moist soil. "
     "Rotate plants for even growth."),

    ("Project Management Tips", 
     "Define scope clearly and prioritize tasks. Short iterations with demos help reduce risk. "
     "Use retrospectives to improve team processes."),

    ("Python Tricks", 
     "List comprehensions are concise. Generators are memory-efficient. "
     "The standard library includes powerful modules like itertools and functools."),

    ("Healthy Sleep", 
     "Consistent sleep schedules align circadian rhythms. Reduce blue light before bed. "
     "A cool, dark, quiet room supports better sleep."),

    ("Running Basics", 
     "Increase weekly mileage gradually to avoid injury. Alternate hard and easy days. "
     "Proper shoes and strength training improve performance."),

    ("Budget Cooking", 
     "Plan meals, buy staples in bulk, and cook once for multiple meals. "
     "Use seasonal produce and freeze leftovers."),

    ("Basic First Aid", 
     "For small cuts, clean the wound and apply a sterile bandage. For burns, cool under running water. "
     "Know when to seek medical help."),

    ("Travel Packing", 
     "Use a packing list and roll clothes to save space. Separate liquids in a clear bag. "
     "Carry essentials like meds and chargers in your personal item.")
]

docs_df = pd.DataFrame(docs, columns=["title", "text"])
docs_df


## 4) Chunking

In [ ]:

def chunk_text(text: str, chunk_size: int, overlap: int) -> List[str]:
    out = []
    start = 0
    n = len(text)
    while start < n:
        end = min(n, start + chunk_size)
        out.append(text[start:end])
        if end == n:
            break
        start = max(0, end - overlap)
    return out

corpus = []
for i, row in docs_df.iterrows():
    chs = chunk_text(row["text"], CHUNK_SIZE, CHUNK_OVERLAP)
    for j, ch in enumerate(chs):
        corpus.append({
            "doc_id": i,
            "doc_title": row["title"],
            "chunk_id": j,
            "text": ch
        })

corpus_df = pd.DataFrame(corpus)
print("Total chunks:", len(corpus_df))
corpus_df.head()


## 5) Embeddings (local)

In [ ]:
# 5) Embeddings (local) + ChromaDB vector store

def embed_local(texts: List[str]) -> np.ndarray:
    # normalize_embeddings=True returns L2-normalized vectors (good for cosine)
    arr = embedder.encode(texts, normalize_embeddings=True)
    return np.asarray(arr, dtype="float32")

# Prepare ids, docs, metadatas
chunk_texts = corpus_df["text"].tolist()
ids = [f"doc{int(r.doc_id)}_chunk{int(r.chunk_id)}" for r in corpus_df.itertuples(index=False)]
metadatas = [
    {"doc_id": int(r.doc_id), "doc_title": str(r.doc_title), "chunk_id": int(r.chunk_id)}
    for r in corpus_df.itertuples(index=False)
]

# Embed once
embeddings = embed_local(chunk_texts)

# Chroma client + collection (persistent)
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

# Recreate collection each run for this PoC (safe for small demos).
# If you want to keep historical data, remove delete_collection.
try:
    chroma_client.delete_collection(name=CHROMA_COLLECTION)
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name=CHROMA_COLLECTION,
    metadata={"hnsw:space": "cosine"},
)

# Add to Chroma (embeddings as python lists)
collection.add(
    ids=ids,
    documents=chunk_texts,
    metadatas=metadatas,
    embeddings=embeddings.tolist(),
)

print("Chroma collection:", CHROMA_COLLECTION)
print("Count:", collection.count())


## 6) Retrieval helpers

In [ ]:
# 6) Retrieval helpers (ChromaDB)

def retrieve(query: str, k: int = TOP_K):
    # Embed query locally (consistent with stored vectors)
    q = embed_local([query])[0].tolist()

    res = collection.query(
        query_embeddings=[q],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    ids = res["ids"][0]
    docs = res["documents"][0]
    metas = res["metadatas"][0]
    dists = res["distances"][0]   # cosine distance (lower is better)

    return ids, docs, metas, dists

def build_context(ids, docs, metas) -> str:
    parts = []
    for _id, doc, meta in zip(ids, docs, metas):
        title = meta.get("doc_title", "Unknown")
        chunk_id = meta.get("chunk_id", "?")
        parts.append(f"[{title}#chunk{chunk_id}] {doc}")
    return "\n\n".join(parts)


## 7) Prompting and RAG answer

In [ ]:


import requests  # ensure requests is available in this cell

# System prompt (global behavior)
SYSTEM_PROMPT = (
    "You are a concise, technical assistant for Amber molecular simulation users. "
    "Answer the user's question using ONLY the provided context. "
    "If the answer cannot be determined from the context, say you do not know. "
    "Cite sources using [Title#chunkN] notation. "
    "Do not speculate or introduce external knowledge."
)

# ---- Ollama helper (must be defined before any caller) ----
def ollama_chat(system_prompt: str, user_prompt: str, model: str = None, url: str = None) -> str:
    """Call Ollama's /api/chat endpoint and return the assistant message content."""
    model = model or OLLAMA_MODEL
    url = (url or OLLAMA_URL).rstrip("/")
    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "stream": False,
    }
    r = requests.post(f"{url}/api/chat", json=payload, timeout=120)
    r.raise_for_status()
    data = r.json()
    # Ollama's response shape may vary by version; try common keys
    # prefer message.content, fallback to text or assistant
    if isinstance(data, dict):
        # common Ollama response: {"message": {"role":"assistant", "content":"..."}}
        msg = data.get("message") or {}
        content = msg.get("content") or msg.get("text") or data.get("text") or ""
        return content
    return str(data)

# ---- Prompt builder ----
def build_user_prompt(query: str, context: str) -> str:
    return (
        "Context:\n" + context + "\n\n"
        "User question: " + query + "\n\n"
        "Instructions:\n"
        "- Use only the context.\n"
        "- If insufficient, say you do not know.\n"
        "- Include short inline citations like [Title#chunkN].\n"
    )

# ---- RAG answer function (calls ollama_chat) ----

def answer_with_rag(query: str, k: int = TOP_K):
    ids, docs, metas, dists = retrieve(query, k=k)
    context = build_context(ids, docs, metas)

    # Optional: cap context length for local LLM speed
    context_for_prompt = context[:2500]
    user_prompt = build_user_prompt(query, context_for_prompt)

    # Optional quick health check
    try:
        _ = requests.get(OLLAMA_URL.rstrip("/") + "/", timeout=3)
    except Exception as e:
        return {
            "query": query,
            "answer": f"Ollama not reachable ({type(e).__name__}: {e}). Returning context summary:\n\n{context_for_prompt}",
            "ids": ids,
            "distances": dists,
            "context": context,
        }

    try:
        answer = ollama_chat(SYSTEM_PROMPT, user_prompt)
    except Exception as e:
        answer = (
            f"Ollama call failed ({type(e).__name__}: {e}). Returning a context summary instead:\n\n"
            + context_for_prompt
        )

    return {
        "query": query,
        "answer": answer,
        "ids": ids,
        "distances": dists,
        "context": context,
    }


In [ ]:
# --- Interactive demo (runs once) ---
example_query = input("Enter your Amber-related question (or press Enter to skip): ").strip()

if not example_query:
    print("No query provided — demo skipped.")
else:
    res = answer_with_rag(example_query)
    print("\nQ:", res["query"])
    print("\n--- Retrieved context ---\n", res["context"][:800])
    print("\n--- Answer ---\n", res["answer"])


## 8) Simple evaluation

In [ ]:
import re
import pandas as pd

citation_pattern = re.compile(r"\[([^\[\]#]+)#chunk(\d+)\]")

def parse_citations(text: str):
    cites = []
    for m in citation_pattern.finditer(text or ""):
        cites.append((m.group(1).strip(), int(m.group(2))))
    return cites

def parse_context_citations(context: str):
    # Extract the set of (title, chunk_id) that were actually provided to the LLM
    return set(parse_citations(context or ""))

def eval_one(qitem):
    out = answer_with_rag(qitem["q"])

    # Expected substring check
    has_expected = qitem["expect_substring"].lower() in (out["answer"] or "").lower()

    # Get citations that were present in the LLM answer
    cited = parse_citations(out["answer"])

    # Get citations that were actually present in the provided context (retrieved chunks)
    context_keys = parse_context_citations(out["context"])
    cited_keys = set(cited)

    # Are all answer citations from the retrieved context?
    citations_within_topk = cited_keys.issubset(context_keys) if cited_keys else True

    # Did it cite the expected source title?
    expected_title_cited = any(t == qitem["expect_title"] for (t, _) in cited)

    return {
        "question": qitem["q"],
        "expected_substring": qitem["expect_substring"],
        "has_expected_substring": has_expected,
        "citations_within_topk": citations_within_topk,
        "expected_title_cited": expected_title_cited,
        "citations": cited,
        "answer": out["answer"],
    }

rows = [eval_one(q) for q in qa_gold]
pd.DataFrame(rows)[["question","has_expected_substring","citations_within_topk","expected_title_cited","citations"]]



## 9) Notes

- This version removes redundant imports, duplicate embedding calls, and OpenAI embedding usage (which was rate-limited).
- Retrieval embeddings for both docs and queries are **local** and consistent (same model).
- Swap `GEN_MODEL` if you want a different chat model.
- To go fully offline, replace the generation step with a local LLM (e.g., via `ollama`).
